# Connecting the MCP Chatbot to Reference Servers

You'll now extend the MCP chatbot capabilities by making it connect to any MCP server. You will integrate the tools of two official MCP servers in addition to the tools of the research server you built in a previous exercise. 

<img src="images/lesson_6.png" width="700">

## Open-Source MCP Servers

In this [repo](https://github.com/modelcontextprotocol/servers), you can find a collection of reference implementations for the MCP servers, as well as references to community built servers and additional resources. You will use two of the reference servers to integrate their tools in your MCP chatbot:
- [fetch](https://github.com/modelcontextprotocol/servers/tree/main/src/fetch): provides the `fetch` tool which fetches a URL from the internet and extracts its contents as markdown.
- [filesystem](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem): provides several tools for interacting with the files and directories within a directory that you specify.

You can check the readme file of each server to check the features they expose and how to run them.  

## Updating the MCP Chatbot - Optional Reading

Here are the updates you'll make to the chatbot. You're encouraged to read this section before or after you watch the video, if you'd like to learn more about the details of the code.

- Instead of hardcoding the server parameters in the chatbot, the chatbot will read the server configurations from a JSON file:
  ### Server Configuration
  In the `4-ConnectingToReference`, you can find the `server_config.json` configuration file that has the following structure.
    ``` json
    {
        "mcpServers": {
            
            "filesystem": {
                "command": "npx",
                "args": [
                    "-y",
                    "@modelcontextprotocol/server-filesystem",
                    "."
                ]
            },
            
            "research": {
                "command": "uv",
                "args": ["run", "../2-MCPServer/research_server.py"]
            },
            
            "fetch": {
                "command": "uvx",
                "args": ["mcp-server-fetch"]
            }
        }
    }
    ```
    For the reference servers, the commands `npx` and `uvx` directly install the files of the servers to your local environment (you don't need to install them ahead of time). Note for the `filesystem`, the `.` is provided as the third argument and it means "current directory". This means that you're allowing for the `fetch` server to interact with the files and directories that are within the current directory.

- For the updated MCP_chatbot:
  

  1. Instead of having one session, you now have a list of client sessions where each client session establishes a 1-to-1 connection to each server;
  2. `available_tools` includes the definitions of all the tools exposed by all servers that the chatbot can connect to.
  3. `tool_to_session` maps the tool name to the corresponding client session; in this way, when the LLM decides on a particular tool name, you can map it to the correct client session so you can use that session to send `tool_call` request to the right MCP server.
  4. `exit_stack` is a context manager that will manage the mcp client objects and their sessions and ensures that they are properly closed. In exercise 3, you did not use it because you used the `with` statement which behind the scenes uses a context manager. Here you could again use the `with` statement, but you may end up using multiple nested `with` statements since you have multiple servers to connect to. `exit_stack` allows you to dynamically add the mcp clients and their sessions as you'll see in the code below.
  5. `connect_to_servers` reads the server configuration file and for each single server, it calls the helper method `connect_to_server`. In this latter method, an MCP client is created and used to launch the server as a sub-process and then a client session is created to connect to the server and get a description of the list of the tools provided by the server.
  6. `cleanup` is a helper method that ensures all your connections are properly shut down when you're done with them. In exercise 3, you relied on the `with` statement to automatically clean up resources. This cleanup method serves a similar purpose, but for all the resources you've added to your exit_stack; it closes (your MCP clients and sessions) in the reverse order they were added - like stacking and unstacking plates. This is particularly important in network programming to avoid resource leaks.

## Updated Code for the MCP Chatbot

In [ ]:
%%writefile mcp_chatbot.py
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from typing import List, Dict, TypedDict
import asyncio
# import nest_asyncio
import json
from gates_openai import create_response
from contextlib import AsyncExitStack #new

# nest_asyncio.apply

class ToolDefinition(TypedDict):
    type: str
    name: str
    description: str
    parameters: dict

class MCP_ChatBot:

    def __init__(self):
        # Initialize session and client objects
        #self.session: ClientSession = None
        self.sessions: List[ClientSession] = [] #new
        self.exit_stack = AsyncExitStack() #new
        # self.available_tools: List[dict] = []
        self.available_tools: List[ToolDefinition] = [] #new
        self.tool_to_session: Dict[str, ClientSession] = {} #new

    async def connect_to_server(self, server_name: str, server_config: dict) -> None:
        """Connect to a single MCP server."""
        try:
            # server_params = StdioServerParameters(
            #     command="uv",  # Executable
            #     args=["run", "research_server.py"],  # Optional command line arguments
            #     env=None,  # Optional environment variables
            # )

            server_params = StdioServerParameters(**server_config)
            stdio_transport = await self.exit_stack.enter_async_context(
                stdio_client(server_params)
            ) #new

            # async with stdio_client(server_params) as (read, write):
            #     async with ClientSession(read, write) as session:
            #         self.session = session
            #         # Initialize the connection
            #         await session.initialize()

            read,write = stdio_transport
            session = await self.exit_stack.enter_async_context(
                ClientSession(read, write)
            ) #new

            await session.initialize()
            self.sessions.append(session)

            # List available tools for this session
            response = await session.list_tools()
            tools = response.tools
            print(f"\nConnected to {server_name} with tools:", [tool.name for tool in tools])
            
            # self.available_tools = [{
            #     "type": "function",
            #     "name": tool.name,
            #     "description": tool.description,
            #     "parameters": tool.inputSchema
            # } for tool in response.tools]
            
            for tool in tools: #new
                self.tool_to_session[tool.name] = session
                self.available_tools.append({
                    "type": "function",
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema
                })

            # await self.chat_loop()
        except Exception as e:
            print(f"Failed to connect to {server_name}: {e}")


    async def connect_to_servers(self): #new
        """Connect to all configured MCP servers."""
        try:
            with open("server_config.json", "r") as file:
                data = json.load(file)
            servers = data.get("mcpServers", {})

            for server_name, server_config in servers.items():
                await self.connect_to_server(server_name, server_config)

        except Exception as e:
            print(f"Error loading server configuration: {e}")
            raise



    async def process_query(self, query: str = None, previous_response_id: str = None):
        
        messages = [
            {
                "role": "system",
                "content": """You are a helpful assistant.
                Use search_papers tool to search for academic papers on a given topic on arxiv.
                User extract_info tool to get more information on papers retrieved through the search_papers tool.
                """
            },
            {
                "role":"user",
                "content":query
            }
        ]
        
        kwargs = {}

        if previous_response_id:
            kwargs["previous_response_id"] = previous_response_id

        response = create_response(
            model = "gpt-4o-mini",
            tools = self.available_tools,
            input = messages,
            **kwargs
        )

        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if not function_calls:
            return response.output_text, response.id

        while True:

            tool_outputs = []

            for call in function_calls:

                tool_name = call.name
                tool_args = json.loads(call.arguments)

                session = self.tool_to_session[tool_name] #new
                result = await session.call_tool(tool_name, arguments = tool_args)
                tool_output = "\n".join(map(str, result))
                # print(tool_output)

                tool_outputs.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": tool_output
                })

            messages.extend(tool_outputs)

            response = create_response(
                model = "gpt-4o-mini",
                input = tool_outputs,
                tools = self.available_tools,
                previous_response_id = response.id
            )


            function_calls = [
                item for item in response.output
                if item.type == "function_call"
            ]

            if function_calls:
                print("Function calls found...")
                continue
            else:
                return response.output_text, response.id

    
    
    async def chat_loop(self):
        print("Type your queries or 'quit' to exit.")
        response_id = None
        while True:
            try:
                query = input("\nQuery: ").strip()
                if query.lower() == 'quit':
                    break
        
                response, response_id = await self.process_query(query, response_id)
                print("\n")
                print(response)
            except Exception as e:
                print(f"\nError: {str(e)}")

    async def cleanup(self): #new
        """Cleanly close all resources using AsyncExitStack"""
        await self.exit_stack.aclose()



async def main():
    """
    The mcp clients and sessions are not initialized using `with` like in the previous exercise
    So cleanup should be manually handled
    """
    chatbot = MCP_ChatBot()
    try:
        # await chatbot.connect_to_server_and_run()
        await chatbot.connect_to_servers() #new
        await chatbot.chat_loop()
    finally:
        await chatbot.cleanup() #new
  

if __name__ == "__main__":
    asyncio.run(main())

## Running the MCP Chatbot

**Terminal Instructions**

- Open a terminal
- Navigate to the `4-ConnectingToReference` directory:
    - `cd 4-ConnectingToReference`
    - `uv init`
- Activate the virtual environment:
    - `uv venv`
    - `source .venv/bin/activate`
- Install the additional dependencies:
    - `uv add mcp arxiv openai`
- Run the chatbot:
    - `uv run mcp_chatbot.py`
- To exit the chatbot, type `quit`.

Make sure to interact with the chatbot. Here are some query examples. You may encounter errors on the first query. That's OK. That's expected if you don't have server-filesystem and mcp-server-fetch pre-installed:
- Fetch the content of this website: https://modelcontextprotocol.io/docs/concepts/architecture and save the content in the file "mcp_summary.md", create a visual diagram that summarizes the content of "mcp_summary.md" and save it in a text file
- Fetch https://developers.openai.com/blog and find an interesting term. Search for 2 papers around the term and then summarize your findings and write them to a file called results.txt